# Mini GPT: from bigram to a full transformer
Continuation of `bigram.ipynb`. Same data, same char-level tokenizer, but now the model can look at **all previous characters** (not just one) using self-attention.

Pieces we add:
1. Positional embeddings
2. Self-attention head (`Head`)
3. Multi-head attention
4. Feed-forward network
5. Transformer `Block` (residual connections + LayerNorm)
6. Full `GPTLanguageModel`, training loop, generation

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

# hyperparameters
batch_size = 32      # sequences processed in parallel
block_size = 128     # context length (how many characters the model sees)
max_iters = 3000
eval_interval = 500
eval_iters = 100
learning_rate = 3e-4
n_embd = 384         # embedding dimension
n_head = 4           # attention heads per block
n_layer = 4          # number of transformer blocks
dropout = 0.2

torch.manual_seed(1337)

mps


---
**Data + tokenizer** (unchanged from the bigram notebook)

In [2]:
with open('sherlock_holmes.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
print(vocab_size)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

88


---
**Self-attention head**  
Every token emits a **query** (what am I looking for?), a **key** (what do I contain?) and a **value** (what I give out if attended to).  
- `q @ k^T` gives affinity scores between every pair of positions  
- scale by `1/sqrt(head_size)` so softmax doesn't saturate  
- the **lower-triangular mask** (`tril`) stops a token from looking at the future, which is what makes this a *decoder*  
- softmax turns scores into weights, then we take a weighted sum of the values  

This is the `torch.tril` / `masked_fill(-inf)` / `softmax` trick from `torch_examples.ipynb`.

In [3]:
class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # buffer = not a learnable parameter, but moves with the model to the device
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        # attention scores, scaled
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)  # (B, T, hs)
        return wei @ v     # (B, T, hs)

---
**Multi-head attention**  
Run several heads in parallel (each looks for different relationships), concatenate their outputs, and project back to `n_embd`.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, num_heads * head_size)
        return self.dropout(self.proj(out))

---
**Feed-forward + Transformer block**  
- Attention = tokens *communicate*. Feed-forward = each token *thinks* on its own.  
- **Residual connections** (`x + ...`) let gradients flow through deep stacks.  
- **LayerNorm** keeps activations well-scaled (applied before each sub-layer, "pre-norm").

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication (attention) followed by computation (MLP)."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

---
**The full GPT model**  
Compared with the bigram model:
- token embedding is now `vocab_size -> n_embd` (not `vocab_size -> vocab_size`)
- a **position embedding** tells the model *where* each token is (attention alone has no sense of order)
- a stack of `n_layer` blocks
- final LayerNorm + `lm_head` projects back to vocab logits
- `generate` crops the context to the last `block_size` tokens, since position embeddings only exist up to that length

In [6]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)                              # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=index.device))  # (T, C)
        x = tok_emb + pos_emb                                                    # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                                 # (B, T, vocab)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]          # crop to the last block_size tokens
            logits, _ = self(index_cond)
            logits = logits[:, -1, :] / temperature      # (B, C)
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index


model = GPTLanguageModel(vocab_size)
m = model.to(device)
print(f'{sum(p.numel() for p in m.parameters()) / 1e6:.2f}M parameters')

7.21M parameters


---
**Training**  
Same loop as the bigram notebook, just evaluating every `eval_interval` steps. Loss should drop from about 4.5 (random guessing, `ln(88)`) to roughly 1.5 or below. On an Apple Silicon GPU, expect some minutes for 3000 iterations.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, train loss: 4.5816, val loss: 4.5770
step: 500, train loss: 1.8360, val loss: 1.8333


---
**Generate text**  
Start from a single newline token (index 0), or give the model your own prompt.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [ ]:
# Custom prompt (characters must exist in the vocabulary)
prompt = 'Holmes looked at me and said, '
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500, temperature=0.8)[0].tolist()))

---
**Save / load the trained model**

In [ ]:
torch.save(model.state_dict(), 'mini_gpt.pt')

# to load later (re-run the config + class cells first):
# model = GPTLanguageModel(vocab_size)
# model.load_state_dict(torch.load('mini_gpt.pt', map_location=device))
# model.to(device).eval()